# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = 'YOUR_TOKEN_HERE'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if hf_token != 'YOUR_TOKEN_HERE':
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Materialize 50k rows for fast evaluation
query = f"""
WITH metrics AS (
    SELECT 
        f.content_hash_id,
        SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as early_imps,
        SUM(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_clicks ELSE 0 END) as early_clicks,
        AVG(CASE WHEN f.report_date <= '2026-03-15' THEN f.gsc_avg_position ELSE NULL END) as early_pos,
        SUM(CASE WHEN f.report_date > '2026-03-15' THEN f.gsc_impressions ELSE 0 END) as late_imps
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    WHERE ga4_data_available = TRUE -- Handle zero-filled data issue natively!
    GROUP BY f.content_hash_id
)
SELECT 
    m.content_hash_id,
    COALESCE(d.word_count, 0) as word_count,
    CASE WHEN d.word_count IS NOT NULL THEN 1 ELSE 0 END as has_word_count,
    m.early_imps,
    m.early_clicks,
    CASE WHEN m.early_imps > 0 THEN (m.early_clicks * 1.0 / m.early_imps) * 100 ELSE 0 END as ctr,
    COALESCE(m.early_pos, 100) as early_pos,
    CASE WHEN (m.late_imps < (m.early_imps * 0.8)) THEN 1 ELSE 0 END as is_declining,
    m.late_imps as trap_late_imps
FROM metrics m
JOIN read_parquet('{REL}/dim_content.parquet') d ON m.content_hash_id = d.content_hash_id
WHERE m.early_imps > 50
LIMIT 50000
"""
df_features = con.execute(query).df()
display(df_features.head())

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature Definitions & Integrity Check:**

1.  **`early_imps` & `early_clicks`**: Sum of impressions/clicks from March 1 to March 15. Known at decision moment.
2.  **`ctr`**: Early clicks / early imps. Engineered from early metrics. Known.
3.  **`early_pos`**: Average position from March 1 to March 15. Missing values (if 0 impressions) are given a penalty value of 100.
4.  **`word_count`**: Number of words in the content. Known as it's a static property. Missing values are filled with 0, but accompanied by `has_word_count`.
5.  **`has_word_count`**: A boolean flag (`1` or `0`). If a content type systematically drops the word count, the model can learn this structural missingness instead of thinking the article is literally 0 words long.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# We simulate the exact trap that analysts fall into when they forget to align windows.

features_trap = ['early_imps', 'early_clicks', 'ctr', 'early_pos', 'word_count', 'has_word_count', 'trap_late_imps']
features_honest = ['early_imps', 'early_clicks', 'ctr', 'early_pos', 'word_count', 'has_word_count']

# --- EXPERIMENT 1: THE LEAKY MODEL ---
X_trap = df_features[features_trap]
y = df_features['is_declining']
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_trap, y, test_size=0.2, random_state=42)

rf_trap = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
rf_trap.fit(Xt_train, yt_train)
trap_auc = roc_auc_score(yt_test, rf_trap.predict_proba(Xt_test)[:, 1])

importances_trap = pd.Series(rf_trap.feature_importances_, index=features_trap).sort_values(ascending=False)

print(f"--- LEAKY MODEL (with trap_late_imps) ---")
print(f"ROC AUC: {trap_auc:.4f}")
print("Feature Importances:")
print(importances_trap)
print("\n")

# --- EXPERIMENT 2: THE HONEST MODEL ---
X_honest = df_features[features_honest]
Xh_train, Xh_test, yh_train, yh_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

rf_honest = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
rf_honest.fit(Xh_train, yh_train)
honest_auc = roc_auc_score(yh_test, rf_honest.predict_proba(Xh_test)[:, 1])

importances_honest = pd.Series(rf_honest.feature_importances_, index=features_honest).sort_values(ascending=False)

print(f"--- HONEST MODEL (trap removed) ---")
print(f"ROC AUC: {honest_auc:.4f}")
print("Feature Importances:")
print(importances_honest)

**Leakage Test Findings:**
When we included `trap_late_imps`, the model's AUC skyrocketed. The feature importance for the trap variable completely dominated every other feature. This happens because `trap_late_imps` is mathematically intertwined with our label `is_declining` (which checks if late imps dropped). Once we drop the trap feature, the ROC AUC plummets back to reality, showing exactly why future information destroys predictive modeling.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Fields refused:**
*   `trend_pct`: Computed directly from the difference between the 90-day window and the current window, meaning it inherently contains data from the outcome period (future information).
*   `trend_direction`: Derived directly from `trend_pct`. Same leakage problem.
*   `client_id` / `content_hash_id`: These are pseudonyms. If we use them as features, a tree-based model might just memorize the specific IDs rather than learning generalizable signals like impressions and position.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.